Samantha Asefi (sma9am@virginia.edu)
DS 5001
8 May 2026

# Word2vec

Exploration of word embeddings

## imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from gensim.models import word2vec
from sklearn.manifold import TSNE
import plotly.express as px

f2_dir = Path('corpus/f2')
f5_dir = Path('corpus/f5')
f5_dir.mkdir(parents=True, exist_ok=True)

vector_size = 246


## data loading

In [2]:
OHCO  = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
CHAPS = OHCO[:2]
BAG   = OHCO[:4]  # sentence-level context for word2vec
window = 5

TOKEN = pd.read_csv(f2_dir / 'TOKEN.csv').set_index(OHCO)
VOCAB = pd.read_csv('corpus/f4/VOCAB.csv', index_col='term_str')

print('TOKEN shape:', TOKEN.shape)
TOKEN.head()

TOKEN shape: (437101, 5)


pos_tuple  \
book_id            chap_num para_num sent_num token_num                            
fourier_selections 1        3        0        2          ('philosophical', 'JJ')   
                                              3                 ('whims', 'NNS')   
                                              4                ('called', 'VBN')   
                                              5                ('duties', 'NNS')   
                                              8               ('relation', 'NN')   

                                                             token_str  pos  \
book_id            chap_num para_num sent_num token_num                       
fourier_selections 1        3        0        2          philosophical   JJ   
                                              3                  whims  NNS   
                                              4                 called  VBN   
                                              5                 duties  NNS   
                                              8               relation   NN   

                                                              term_str  \
book_id            chap_num para_num sent_num token_num                  
fourier_selections 1        3        0        2          philosophical   
                                              3                  whims   
                                              4                 called   
                                              5                 duties   
                                              8               relation   

                                                         is_stop  
book_id            chap_num para_num sent_num token_num           
fourier_selections 1        3        0        2            False  
                                              3            False  
                                              4            False  
                                              5            False  
                                              8            False

In [3]:
corpus = (
    TOKEN[TOKEN.pos.notna() & ~TOKEN.pos.str.match('NNPS?')]
    .groupby(BAG)
    .term_str.apply(lambda x: [t for t in x.tolist() if isinstance(t, str) and t == t])
    .reset_index()['term_str']
    .tolist()
)

corpus building: when building the corpus, d

## Generate word embeddings with Gensim's library

In [4]:
model = word2vec.Word2Vec(corpus,vector_size=246,window=window,min_count=5,workers=4,epochs=10,seed=42)


In [5]:
print(f'Vocabulary size: {len(model.wv)}')


Vocabulary size: 3985


# Visualize with tSNE

In [6]:
def make_tsne_df(model):
    words = list(model.wv.index_to_key)
    vectors = np.array([model.wv[word] for word in words])

    tsne = TSNE(
        perplexity=40,
        n_components=2,
        init='pca',
        random_state=23,
        max_iter=2500
    )

    tsne_values = tsne.fit_transform(vectors)

    coords = pd.DataFrame({
        'label': words,
        'x': tsne_values[:, 0],
        'y': tsne_values[:, 1]
    })

    return coords

coords = make_tsne_df(model)

In [7]:
fig = px.scatter(
    coords, x='x', y='y',
    text='label',
    title='Word2Vec Embeddings — t-SNE Projection',
    width=1000, height=700
)
fig.update_traces(textposition='top center', marker=dict(size=4))
fig.update_layout(showlegend=False)
fig.show()


In [8]:
# Build embeddings dataframe indexed by term_str
w2v_words   = list(model.wv.index_to_key)
w2v_vectors = np.array([model.wv[w] for w in w2v_words])

W2V = pd.DataFrame(
    w2v_vectors,
    index=w2v_words,
    columns=[f'w2v_{i:03d}' for i in range(vector_size)]
)
W2V.index.name = 'term_str'

# Save standalone embeddings table
W2V.to_csv(f5_dir / 'W2V.csv')

# Append to VOCAB_F5 — reads what LDA wrote, adds w2v columns
VOCAB_F5 = pd.read_csv(f5_dir / 'VOCAB_F5.csv', index_col='term_str')
VOCAB_F5 = VOCAB_F5.join(W2V, how='left')
VOCAB_F5.to_csv(f5_dir / 'VOCAB_F5.csv')

print('Saved to corpus/f5/:')
for f in sorted(f5_dir.iterdir()):
    print(' ', f.name)
print()
print('W2V.csv       — embeddings:', W2V.shape)
print('VOCAB_F5.csv  — vocab + PCA + LDA + w2v:', VOCAB_F5.shape)


Saved to corpus/f5/:
  LDA_DOC_TOPIC.csv
  PCA_DCM_chaps.csv
  PCA_TCM_chaps.csv
  VOCAB_F5.csv
  W2V.csv
  lda_theta_heatmap.png
  lda_top_words.png

W2V.csv       — embeddings: (3985, 246)
VOCAB_F5.csv  — vocab + PCA + LDA + w2v: (9668, 279)
